# 第 5 章 · 持久化、流式与人机协同：LangGraph 的三大超能力

> 第 4 章的图是"一次性"的——跑完就忘。本章给图装上三样生产级装备：
> 1. **Checkpointer**：每个节点执行后自动存档 → 记忆、崩溃恢复、时间旅行；
> 2. **流式输出**：三种 stream_mode，从"整段状态"到"逐 token"；
> 3. **interrupt + Command**：人机协同——让图暂停下来，等人拍板后再继续。

---

## 1. Checkpointer：图的"自动存档"

```mermaid
flowchart LR
    N1["节点1"] -->|"✅ checkpoint#1"| CP[("Checkpointer<br/>存档")]
    N2["节点2"] -->|"✅ checkpoint#2"| CP
    N3["节点3"] -->|"✅ checkpoint#3"| CP
    CP -.->|"get_state / get_state_history<br/>读取 / 回放 / 分叉"| YOU["你"]
    style CP fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

| 实现 | 用途 |
|---|---|
| `InMemorySaver` | 学习与测试（进程内存） |
| `SqliteSaver` | 单机持久化 |
| `PostgresSaver` | 生产环境 |

两个关键概念：

- **thread（线程）**：一段连续对话/任务的标识，≈ ADK 的 Session。通过 `config = {"configurable": {"thread_id": "..."}}` 指定；
- **checkpoint（检查点）**：thread 内每执行完一个节点存一次档，带 `checkpoint_id`——这是时间旅行的坐标。


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# 一个"记账"图：每步往流水账里追加一条
class LedgerState(TypedDict):
    balance: int
    log: Annotated[list[str], add]

def deposit(state: LedgerState) -> dict:
    return {"balance": state["balance"] + 100, "log": [f"存入100 → 余额{state['balance'] + 100}"]}

def withdraw(state: LedgerState) -> dict:
    return {"balance": state["balance"] - 30, "log": [f"取出30 → 余额{state['balance'] - 30}"]}

b = StateGraph(LedgerState)
b.add_node("deposit", deposit)
b.add_node("withdraw", withdraw)
b.add_edge(START, "deposit")
b.add_edge("deposit", "withdraw")
b.add_edge("withdraw", END)

checkpointer = InMemorySaver()
ledger = b.compile(checkpointer=checkpointer)   # ← 挂上存档器

config = {"configurable": {"thread_id": "account-001"}}
result = ledger.invoke({"balance": 0, "log": []}, config)
print("最终状态：", result)


最终状态： {'balance': 70, 'log': ['存入100 → 余额100', '取出30 → 余额70']}


---

## 2. 时间旅行：读取历史检查点

挂了 Checkpointer 后，整条执行轨迹都可回溯：


In [2]:
# 查看当前状态（含 checkpoint 元信息）
snap = ledger.get_state(config)
print("当前余额：", snap.values["balance"], "| checkpoint:", snap.config["configurable"]["checkpoint_id"][:8], "...")
print("下一步待执行节点：", snap.next or "（已结束）")
print("─" * 55)

# 查看全部历史（时间倒流）
print("📜 执行轨迹回放（新→旧）：")
for s in ledger.get_state_history(config):
    print(f"  节点后状态: {s.next or ['END']} | balance={s.values.get('balance')} | log={s.values.get('log')}")


当前余额： 70 | checkpoint: 1f19fcd1 ...
下一步待执行节点： （已结束）
───────────────────────────────────────────────────────
📜 执行轨迹回放（新→旧）：
  节点后状态: ['END'] | balance=70 | log=['存入100 → 余额100', '取出30 → 余额70']
  节点后状态: ('withdraw',) | balance=100 | log=['存入100 → 余额100']
  节点后状态: ('deposit',) | balance=0 | log=[]
  节点后状态: ('__start__',) | balance=None | log=[]


In [3]:
# 分叉（fork）：回到历史某个点，改写状态，走出一条"平行世界线"
history = list(ledger.get_state_history(config))
past = history[1]   # 取一个中间检查点（deposit 之后、withdraw 之前）
print("回到过去：", past.values)

# 在那个时点修改状态：把余额改成 999
fork_config = ledger.update_state(past.config, {"balance": 999, "log": ["💥 平行世界：余额被改为999"]})
forked = ledger.invoke(None, fork_config)   # 从分叉点继续执行（withdraw 节点）
print("平行世界结局：", forked)


回到过去： {'balance': 100, 'log': ['存入100 → 余额100']}
平行世界结局： {'balance': 969, 'log': ['存入100 → 余额100', '💥 平行世界：余额被改为999', '取出30 → 余额969']}


> 🔍 原时间线余额 70，分叉时间线余额 969——**同一张图，两条世界线**。这就是 LangGraph 著名的 "time travel"：调试（回到出错前改输入重跑）、探索（What-if 分析）、纠错（改写 Agent 的错误决策后让它继续）都靠它。

---

## 3. 流式输出：三种颗粒度

| stream_mode | 产出内容 | 适用 |
|---|---|---|
| `"values"` | 每个节点完成后的**完整 State** | 观察状态演化 |
| `"updates"` | 每个节点的**增量更新** | 追踪哪个节点改了什么 |
| `"messages"` | **逐 token** 的 LLM 输出 | 聊天界面的打字机效果 |

用第 4 章的手工 ReAct Agent 来对比感受：


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

llm = ChatOpenAI(model="deepseek-chat", api_key=os.environ["DEEPSEEK_API_KEY"],
                 base_url="https://api.deepseek.com", temperature=0)

@tool
def get_weather(city: str) -> dict:
    """查询城市天气。

    Args:
        city: 城市名。
    """
    return {"city": city, "weather": "晴", "temp": "25°C"}

def call_model(state: MessagesState) -> dict:
    return {"messages": [llm.bind_tools([get_weather]).invoke(state["messages"])]}

b = StateGraph(MessagesState)
b.add_node("call_model", call_model)
b.add_node("tools", ToolNode([get_weather]))
b.add_edge(START, "call_model")
b.add_conditional_edges("call_model", tools_condition)
b.add_edge("tools", "call_model")
agent = b.compile()

# values 模式：每个节点后打印完整状态的消息数
print("── stream_mode='values' ──")
for chunk in agent.stream({"messages": [HumanMessage(content="北京天气如何？")]}, stream_mode="values"):
    print(f"  消息数 → {len(chunk['messages'])} | 最新: {chunk['messages'][-1].type}")


── stream_mode='values' ──
  消息数 → 1 | 最新: human


  消息数 → 2 | 最新: ai
  消息数 → 3 | 最新: tool


  消息数 → 4 | 最新: ai


In [5]:
# messages 模式：逐 token（打字机效果）
print("── stream_mode='messages' ──")
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="用一句话夸夸晴天")]},
    stream_mode="messages",
):
    if token.content:
        print(token.content, end="", flush=True)
print()


── stream_mode='messages' ──


晴天

真是

让人

心情

愉悦

的好

天气

！

> 💡 `metadata` 里带着 token 产自哪个节点——多 Agent 系统里可以按节点过滤流式输出（比如只把"最终回复 Agent"的 token 推给前端）。

---

## 4. 人机协同：interrupt 与 Command

LangGraph 1.x 的人机协同是一对原语：

- **`interrupt(payload)`**：在节点内调用 → 图**立即暂停**并存档，payload 抛给调用方（"我要做什么，请批示"）；
- **`Command(resume=value)`**：再次 invoke 时传入 → 图从暂停点**原地复活**，`interrupt()` 的返回值就是 resume 的值。

**场景**：财务 Agent 付款前必须人工审批。

```mermaid
sequenceDiagram
    participant U as 调用方
    participant G as 图
    participant H as 人类审批者
    U->>G: invoke(付款请求)
    G->>G: 执行到审批节点 interrupt()
    G-->>U: 返回 __interrupt__（待审批信息）
    U->>H: 展示给人类
    H-->>U: 批准 / 拒绝
    U->>G: invoke(Command(resume=决定))
    G->>G: 从断点继续执行
    G-->>U: 最终结果
```


In [6]:
from langgraph.types import interrupt, Command

class PayState(TypedDict):
    item: str
    amount: int
    status: str

def prepare(state: PayState) -> dict:
    return {"status": f"已生成订单：{state['item']}，¥{state['amount']}"}

def human_approval(state: PayState) -> dict:
    # 暂停图执行，把审批请求抛出去；恢复时 resume 的值就是这里的返回值
    decision = interrupt({"待审批": state["status"], "请决定": "approve 或 reject"})
    if decision == "approve":
        return {"status": state["status"] + " → ✅ 审批通过，已付款"}
    return {"status": state["status"] + " → ❌ 审批拒绝，订单取消"}

b = StateGraph(PayState)
b.add_node("prepare", prepare)
b.add_node("human_approval", human_approval)
b.add_edge(START, "prepare")
b.add_edge("prepare", "human_approval")
b.add_edge("human_approval", END)

payment = b.compile(checkpointer=InMemorySaver())   # interrupt 必须有 checkpointer
cfg = {"configurable": {"thread_id": "pay-001"}}

# 第一次调用：图会在审批节点暂停
r = payment.invoke({"item": "机械键盘", "amount": 899, "status": ""}, cfg)
print("⏸️ 图已暂停，待办：", r["__interrupt__"][0].value)


⏸️ 图已暂停，待办： {'待审批': '已生成订单：机械键盘，¥899', '请决定': 'approve 或 reject'}


In [7]:
# 人类看完待办后做出决定 → 用 Command(resume=...) 让图继续
r = payment.invoke(Command(resume="approve"), cfg)
print("▶️ 恢复执行，最终结果：", r["status"])

# 换个线程演示"拒绝"分支
cfg2 = {"configurable": {"thread_id": "pay-002"}}
payment.invoke({"item": "游戏机", "amount": 3999, "status": ""}, cfg2)
r2 = payment.invoke(Command(resume="reject"), cfg2)
print("▶️ 另一笔订单：", r2["status"])


▶️ 恢复执行，最终结果： 已生成订单：机械键盘，¥899 → ✅ 审批通过，已付款
▶️ 另一笔订单： 已生成订单：游戏机，¥3999 → ❌ 审批拒绝，订单取消


> 🎓 **工程意义**：暂停期间整个现场（所有状态、执行位置）都在 Checkpointer 里——人类可以**几小时后**、甚至**换台机器**来审批，图原地复活。这就是"持久化执行（durable execution）"，也是 LangGraph 区别于"内存 while 循环"式 Agent 的核心壁垒。

> 💡 与 ADK 对照：ADK 用 `LongRunningFunctionTool` + 确认事件实现类似的人机协同；LangGraph 的 interrupt/Command 更细粒度——可以插在图的任意位置，而不只是工具调用点。

---

## 5. 跨线程长期记忆：Store

Checkpointer 管"单个 thread 内"的记忆；跨 thread 的共享记忆（如用户画像）用 **Store**：

| 构件 | 范围 | ADK 对应 |
|---|---|---|
| Checkpointer | thread 内 | Session |
| **Store**（`InMemoryStore` / Postgres） | **跨 thread** | MemoryService / `user:` 前缀 State |

```python
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
store.put(("users", "u001"), "profile", {"爱好": "徒步"})   # (命名空间, 键, 值)
store.search(("users", "u001"))                              # 检索
```

节点函数通过 `store` 参数注入使用。概念与 ADK 第 5 章的 Memory 层完全同构，不再展开实操。

---

## 6. 与 ADK 对照 🔄

| LangGraph | ADK | 差异点评 |
|---|---|---|
| Checkpointer / thread_id | SessionService / session_id | 对应工整；LangGraph 的 checkpoint 粒度更细（节点级） |
| get_state_history + update_state | 无直接对应 | **LangGraph 独家**：时间旅行与分叉 |
| interrupt + Command(resume) | LongRunningFunctionTool + 确认 | LangGraph 可在任意节点暂停 |
| stream_mode 三档 | run_async 事件流（一档） | LangGraph 颗粒度可选 |
| Store | MemoryService | 概念同构 |

---

## 📌 本章要点回顾

- `compile(checkpointer=...)` 一行获得：多轮记忆 + 崩溃恢复 + **时间旅行**；
- `get_state_history` 回放轨迹，`update_state` 分叉平行世界线；
- 流式三档：`values`（全量）/ `updates`（增量）/ `messages`（逐 token）；
- `interrupt()` + `Command(resume=...)` = 生产级人机协同，依托持久化执行；
- Store 负责跨线程长期记忆，与 Checkpointer 互补。

> ➡️ 下一章：[06-多智能体模式与生态对比](06-多智能体模式与生态对比.ipynb) —— 收官章：多 Agent 架构模式、LangSmith 与两大框架终极大对照。
